# V3 GraphRAG over OpenFDA labels
No LLM except (a) one optional meaning-split call in symptom mode and (b) the answer call. Reuses the V1 index (chunks, shards, FAISS). **No re-embedding.**
Flags in `FLAGS` give the ablation rows: alias, rerank, split (`llm` / `cluster` / `none`), few_shot, verify. Set `LLM_MODEL` to the LoRA model name for the LoRA rows.

In [ ]:
# ===== 1. packages + Ollama =====
import os, subprocess, time, requests, shutil
os.environ["JAX_PLATFORMS"] = "cpu"; os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True)
print("pip rc:", sh("pip install -q ollama faiss-cpu bm25s networkx pandas numpy tqdm requests sentence-transformers rapidfuzz scikit-learn").returncode)
if not shutil.which("ollama"):
    sh("apt-get update -qq && apt-get install -y -qq zstd"); sh("curl -fsSL https://ollama.com/install.sh | sh")
def ollama_up():
    try: return requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200
    except Exception: return False
if not ollama_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up(): break
        time.sleep(1)
assert ollama_up(), "Ollama did not start"
for m in ["llama3:8b", "nomic-embed-text"]:
    print(m, "pull rc =", sh(f"ollama pull {m}").returncode)

pip rc: 0
llama3:8b pull rc = 0
nomic-embed-text pull rc = 0


In [ ]:
# ===== 2. config =====
import re, json, pickle, glob, hashlib, html, traceback
from collections import defaultdict, Counter
import numpy as np, pandas as pd, faiss, bm25s
from tqdm.auto import tqdm
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception: print("not in Colab")
assert any(l.split()[1] == "/content/drive" for l in open("/proc/mounts")), "Drive is NOT mounted"

BASE_DIR = "/content/drive/MyDrive/Research_kgTxagent/graph_rag"
OUT_DIR  = os.path.join(BASE_DIR, "Adv_graphRAG")
GRAPH_PATH = next(p for p in ["/content/drive/MyDrive/Research_kgTxagent/Kg_dev_files_and_data/test_openfda/OpenFDA_graph.pkl",
                              os.path.join(BASE_DIR, "OpenFDA_graph.pkl")] if os.path.exists(p))
CHUNK_PATH = sorted(glob.glob(os.path.join(OUT_DIR, "chunks_*.pkl")), key=os.path.getmtime)[-1]
SIG = os.path.basename(CHUNK_PATH)[len("chunks_"):-len(".pkl")]
SHARD_DIR = os.path.join(OUT_DIR, f"shards_{SIG}"); FAISS_PATH = os.path.join(OUT_DIR, f"faiss_{SIG}.index")
QA_PATH = os.path.join(BASE_DIR, "QA.json")
RESULTS_PATH = os.path.join(OUT_DIR, "qa_results_v3.json")

LLM_MODEL, EMBED_MODEL = "llama3:8b", "nomic-embed-text"     # LoRA rows: change LLM_MODEL to the adapter model name
EMBED_URL, GEN_URL = "http://localhost:11434/api/embed", "http://localhost:11434/api/generate"
RERANK_MODEL, RERANK_DEVICE = "BAAI/bge-reranker-base", "cpu"

# ---- ablation switches ----
FLAGS = {"alias": True, "rerank": True, "split": "llm", "few_shot": True, "shot4": False, "verify": True}
# ---- budgets (config; decide after measuring prompt_tokens in the results) ----
NUM_CTX, ANSWER_TOKENS = 8192, 700
CONTEXT_TOKENS, CHARS_PER_TOKEN = 4500, 4      # raised from 3000; auto-shrinks if the prompt would not fit NUM_CTX
POOL, TOP_PASS = 15, 8            # candidate drugs searched / drugs passed to the LLM (symptom path)
MAX_NAMED, PER_NAME = 4, 2        # named-drug path: max drugs, max labels per matched name
SEARCH_K, RERANK_N = 60, 60
CLUSTER_GAP = 2.0                 # rerank-score gap under which two clusters are both kept
PRECISION_MIN, ALIAS_MAX, STOP_DF = 0.15, 3, 300
EXCLUDE_SECTIONS = {"inactive_ingredient"}
CORE_SECTIONS = ["indications_and_usage", "warnings", "do_not_use", "pregnancy_or_breast_feeding", "stop_use"]
RRF_K = 60
MAX_CHUNKS_PER_SECTION = 2
ALIAS_CAP = 2                     # max alias-matched drugs per question
ABSTAIN_TEXT = "INSUFFICIENT FDA DATA: The retrieved label text does not contain this information."
print("SIG:", SIG)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SIG: 7c9ff259


In [ ]:
# ===== 3. load graph, chunks, embeddings, FAISS, BM25 =====
t0 = time.time()
with open(GRAPH_PATH, "rb") as f: G = pickle.load(f)
type_counts = Counter(d.get("type") for _, d in G.nodes(data=True))
SECTION_TYPES = sorted(t for t in type_counts if t not in ("drug_generic", "drug_brand"))
generic_ids = [n for n, d in G.nodes(data=True) if d.get("type") == "drug_generic"]
brand_ids   = [n for n, d in G.nodes(data=True) if d.get("type") == "drug_brand"]
GENERIC_NAME = {n: G.nodes[n].get("name", "") for n in generic_ids}
BRAND_NAME   = {n: G.nodes[n].get("name", "") for n in brand_ids}
BRAND_TO_GENERIC, DRUG_SECTIONS = defaultdict(list), defaultdict(list)
for g in tqdm(generic_ids, desc="drug->section edges"):          # only drug->section edges are used
    for nb in G.neighbors(g):
        t = G.nodes[nb].get("type")
        if t == "drug_brand": BRAND_TO_GENERIC[nb].append(g)
        elif t in SECTION_TYPES: DRUG_SECTIONS[g].append((nb, t))
print(f"graph {time.time()-t0:.0f}s | generics={len(generic_ids):,} brands={len(brand_ids):,}")

def section_text(sid):
    txt = G.nodes[sid].get("description") or ""
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", txt))).strip()

CHUNKS = pickle.load(open(CHUNK_PATH, "rb"))
EMB = np.ascontiguousarray(np.vstack([np.load(p) for p in sorted(glob.glob(os.path.join(SHARD_DIR, "emb_*.npy")))]), dtype="float32")
assert EMB.shape[0] == len(CHUNKS)
faiss.normalize_L2(EMB)
INDEX = faiss.read_index(FAISS_PATH) if os.path.exists(FAISS_PATH) else None
if INDEX is None: INDEX = faiss.IndexFlatIP(EMB.shape[1]); INDEX.add(EMB)

STOP = {"the","a","an","of","and","to","is","are","can","i","if","for","in","on","or","with","what","do","how","my","me","it"}
def tokenize(s):
    s = re.sub(r"(\d)([a-z])", r"\1 \2", s.lower())
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in STOP]
BM25 = bm25s.BM25(); BM25.index([tokenize(c["stamped"]) for c in CHUNKS], show_progress=False)
IDX_BY_SECTION, IDX_BY_DRUG = defaultdict(list), defaultdict(list)
for i, c in enumerate(CHUNKS):
    IDX_BY_SECTION[c["section"]].append(i); IDX_BY_DRUG[c["drug_id"]].append(i)
IDX_BY_SECTION = {k: np.array(v) for k, v in IDX_BY_SECTION.items()}
IDX_BY_DRUG = {k: np.array(v) for k, v in IDX_BY_DRUG.items()}
print("chunks:", len(CHUNKS), "| EMB:", EMB.shape, "| FAISS:", INDEX.ntotal)

drug->section edges:   0%|          | 0/10840 [00:00<?, ?it/s]

graph 6s | generics=10,840 brands=31,797


DEBUG:bm25s:Building index from tokens


chunks: 97579 | EMB: (97579, 768) | FAISS: 97579


In [ ]:
# ===== 4. LLM + embedding helpers, hybrid search =====
def _embed_call(texts):
    r = requests.post(EMBED_URL, json={"model": EMBED_MODEL, "input": texts, "truncate": True}, timeout=300)
    r.raise_for_status(); return r.json()["embeddings"]
_QCACHE = {}
def qvec(q):
    if q not in _QCACHE:
        v = np.array(_embed_call(["search_query: " + q])[0], dtype="float32"); _QCACHE[q] = v / (np.linalg.norm(v) + 1e-9)
    return _QCACHE[q]

LAST_PROMPT_TOKENS = [0]
def llm(prompt, temperature=0.1, num_predict=ANSWER_TOKENS, json_mode=False, tries=3):
    p = {"model": LLM_MODEL, "prompt": prompt, "stream": False, "keep_alive": "30m",
         "options": {"temperature": temperature, "num_ctx": NUM_CTX, "num_predict": num_predict}}
    if json_mode: p["format"] = "json"
    for k in range(tries):
        try:
            r = requests.post(GEN_URL, json=p, timeout=900); r.raise_for_status(); j = r.json()
            LAST_PROMPT_TOKENS[0] = j.get("prompt_eval_count", 0)
            return j["response"].strip()
        except Exception as e:
            print(f"  llm retry {k+1}/{tries}: {repr(e)[:80]}"); time.sleep(5 * (k + 1))
    return ""
def llm_json(prompt, **kw):
    raw = llm(prompt, json_mode=True, num_predict=200, **kw)
    try: return json.loads(raw)
    except Exception:
        m = re.search(r"\{.*\}", raw, re.S)
        try: return json.loads(m.group(0)) if m else None
        except Exception: return None

def dense_topk(qv, allowed, k):
    if allowed is None:
        _, I = INDEX.search(qv.reshape(1, -1).astype("float32"), k); return [int(i) for i in I[0] if i >= 0]
    sims = EMB[allowed] @ qv
    return [int(allowed[o]) for o in np.argsort(-sims)[:k]]
def bm25_topk(q, allowed, k):
    toks = tokenize(q)
    if not toks: return []
    scores = BM25.get_scores(toks)
    if allowed is None:
        cand = np.argpartition(-scores, min(k, len(scores) - 1))[:k]; cand = cand[np.argsort(-scores[cand])]
    else: cand = allowed[np.argsort(-scores[allowed])[:k]]
    return [int(c) for c in cand if scores[c] > 0]
def hybrid_rank(queries, allowed=None, k=50):
    out = defaultdict(float)
    for q in queries:
        for lst in (dense_topk(qvec(q), allowed, k), bm25_topk(q, allowed, k)):
            for rank, idx in enumerate(lst, 1): out[idx] += 1.0 / (RRF_K + rank)
    return sorted(out.items(), key=lambda x: -x[1])[:k]

RERANKER = None
def get_reranker():
    global RERANKER
    if RERANKER is None:
        from sentence_transformers import CrossEncoder
        RERANKER = CrossEncoder(RERANK_MODEL, max_length=512, device=RERANK_DEVICE)
    return RERANKER
print("LLM test:", llm("Reply with the single word OK.", num_predict=5))

LLM test: OK


In [ ]:
# ===== 5. entity table + alias table (data-driven, with guards) =====
def norm_tokens(s):
    s = re.sub(r"(\d)([a-z])", r"\1 \2", s.lower())
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in {"and", "the", "of", "with"}]
squash = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())

name_vocab = defaultdict(set)
for g, nm in GENERIC_NAME.items():
    for t in set(norm_tokens(nm)): name_vocab[t].add(g)
for b, nm in BRAND_NAME.items():
    for t in set(norm_tokens(nm)): name_vocab[t].update(BRAND_TO_GENERIC.get(b, []))

drug_tokens = defaultdict(set)
for c in tqdm(CHUNKS, desc="tokens per drug"): drug_tokens[c["drug_id"]].update(norm_tokens(c["text"]))
DF = Counter()
for d, ts in drug_tokens.items(): DF.update(ts)
text_vocab = defaultdict(set)
for d, ts in drug_tokens.items():
    for t in ts & name_vocab.keys(): text_vocab[t].add(d)

TOK_PREC = {t: (len(nd & text_vocab[t]) / len(text_vocab[t])) if text_vocab.get(t) else 1.0 for t, nd in name_vocab.items()}
ENT, dropped = [], []
for kind, table in (("generic", GENERIC_NAME), ("brand", BRAND_NAME)):
    for eid, nm in table.items():
        if kind == "brand" and not BRAND_TO_GENERIC.get(eid): continue
        ts = frozenset(norm_tokens(nm))
        if not ts: continue
        (ENT if max(TOK_PREC[t] for t in ts) >= PRECISION_MIN else dropped).append((ts, nm, kind, eid))
print(f"entities kept={len(ENT):,} | dropped as category words={len(dropped):,}")
print("sample dropped:", [d[1] for d in dropped[:15]])

# alias table: rare product-like words that appear only inside label text
QSTOP = {"compare","versus","difference","different","products","product","safe","use","used","using","what","which",
         "depending","whether","between","child","children","pregnant","pregnancy","warning","warnings"}
STOP_DATA = {t for t, c in DF.items() if c > STOP_DF}
dom_re = re.compile(r"\b([A-Za-z0-9][A-Za-z0-9\-]{2,})\.(?:com|net|org|co)\b")
tm_re  = re.compile(r"\b([A-Z][A-Za-z0-9\-]{3,})\s?(?:\u2122|\u00ae|\(TM\)|TM\b)")
cap_re = re.compile(r"\b[A-Z][a-z]{4,}[A-Za-z\-]*\b")
alias_dom, cap_hits = defaultdict(set), defaultdict(set)
for c in tqdm(CHUNKS, desc="alias scan"):
    d, t = c["drug_id"], c["text"]
    for m in dom_re.findall(t) + tm_re.findall(t): alias_dom[squash(m)].add(d)
    for m in set(cap_re.findall(t)): cap_hits[squash(m)].add(d)
BAD = QSTOP | STOP_DATA
ALIAS = defaultdict(set)
for a, ds in alias_dom.items():
    if 4 <= len(a) and len(ds) <= ALIAS_MAX and a not in BAD: ALIAS[a] |= ds
for a, ds in cap_hits.items():
    if len(a) >= 5 and len(ds) <= ALIAS_MAX and a not in name_vocab and a not in BAD and DF.get(a, 0) <= 6: ALIAS[a] |= ds
print("alias entries:", len(ALIAS))

def alias_hits(query):
    toks = [squash(t) for t in re.split(r"[\s/,?;:]+", query) if t]
    cand = {t for t in toks if len(t) >= 4} | {a + b for a, b in zip(toks, toks[1:]) if a and b}
    return [{"drug_id": g, "matched": f"alias:{a}", "n": 99} for a in cand if a not in BAD for g in ALIAS.get(a, ())]

def detect_entities(query):
    qn = " ".join(norm_tokens(query)); qt = set(qn.split()); hits = []
    exact, fuzzy = [], []
    for ts, nm, kind, eid in ENT:
        if ts <= qt: exact.append((ts, nm, kind, eid))
        elif len(ts) >= 5 and len(ts) - len(ts & qt) == 1: fuzzy.append((ts, nm, kind, eid))   # fallback only: long name, exactly 1 word missing
    hits = exact if exact else sorted(fuzzy, key=lambda h: -len(h[0]))[:1]
    qp = " " + qn + " "
    def spans(nm):
        ph, out, s = " " + " ".join(norm_tokens(nm)) + " ", [], 0
        while True:
            i = qp.find(ph, s)
            if i < 0: return out
            out.append((i, i + len(ph))); s = i + 1
    sp = {h[1]: spans(h[1]) for h in hits}
    keep = []
    for h in hits:                       # keep a shorter name only if it also appears outside the longer name's span
        longer = [o for o in hits if h[0] < o[0]]
        if not longer: keep.append(h); continue
        cover = [x for o in longer for x in sp[o[1]]]
        if [x for x in sp[h[1]] if not any(a <= x[0] and x[1] <= b for a, b in cover)]: keep.append(h)
    out = []
    for ts, nm, kind, eid in sorted(keep, key=lambda h: (-len(h[0]), len(h[1]))):
        for g in ([eid] if kind == "generic" else BRAND_TO_GENERIC.get(eid, [])): out.append({"drug_id": g, "matched": nm, "n": len(ts), "kind": "exact" if ts <= qt else "fuzzy"})
    if FLAGS["alias"]:
        skip = set().union(*[h[0] for h in keep]) if keep else set()
        al, seen_al = [], set()
        for h in alias_hits(query):
            if h["matched"][6:] in skip or h["drug_id"] in seen_al: continue    # skip aliases already inside a matched name
            seen_al.add(h["drug_id"]); al.append(dict(h, kind="alias"))
        out += al[:ALIAS_CAP]
    return out

_LH = {}
def label_hash(d):
    if d not in _LH:
        txt = "||".join(sorted(section_text(sid).lower() for sid, t in DRUG_SECTIONS[d] if t not in EXCLUDE_SECTIONS))
        _LH[d] = hashlib.md5(txt.encode()).hexdigest()
    return _LH[d]
LAST_HITS = []
def named_drugs(query):
    per, order, seen_h = Counter(), [], set()
    hs = detect_entities(query)
    LAST_HITS.clear(); LAST_HITS.extend((h["matched"], h["kind"], GENERIC_NAME[h["drug_id"]]) for h in hs[:12])
    for h in hs:
        if per[h["matched"]] >= PER_NAME: continue
        hh = label_hash(h["drug_id"])
        if hh in seen_h or h["drug_id"] in order: continue
        seen_h.add(hh); per[h["matched"]] += 1; order.append(h["drug_id"])
        if len(order) >= MAX_NAMED: break
    return order
print("test:", [GENERIC_NAME[d] for d in named_drugs("Can I use Kank-a on my 1-year-old's gums for teething?")],
      [GENERIC_NAME[d] for d in named_drugs("Compare the indications and purpose of Acetaminophen 500 mg with Caffeine 65 mg versus Acetaminophen 500 mg.")])

tokens per drug:   0%|          | 0/97579 [00:00<?, ?it/s]

entities kept=39,939 | dropped as category words=2,695
sample dropped: ['ALCOHOL', 'SKIN PROTECTANT', 'ETHYL ALCOHOL', 'ALUMINUM ZIRCONIUM TETRACHLOROHYDREX GLY', 'ALUMINUM CHLOROHYDRATE', 'DYSPHANIA AMBROSIOIDES', 'ALUMINUM SESQUICHLOROHYDRATE', 'CALMING DAY', 'ETHYL ALCOHOL 70%', 'ANTIBACTERIAL WIPE', 'BUTTERFLY PEA', 'ETHYL ALCOHOL 66.5%', 'CHLORPROMAZINE', 'ALCOHOL 80%W/W', 'GOLD']


alias scan:   0%|          | 0/97579 [00:00<?, ?it/s]

alias entries: 3127
test: ['BENZOCAINE AND MENTHOL, UNSPECIFIED FORM'] ['ACETAMINOPHEN 500 MG WITH CAFFEINE 65MG', 'ACETAMINOPHEN 500MG', 'ACETAMINOPHEN 500 MG', 'ACETAMINOPHEN']


In [ ]:
# ===== 5b. entity-detection tests (overlap, brands, aliases, category words) =====
TESTS = [("Compare the indications and purpose of Acetaminophen 500 mg with Caffeine 65 mg versus Acetaminophen 500 mg.", ["ACETAMINOPHEN 500 MG WITH CAFFEINE 65MG", "ACETAMINOPHEN 500 MG"]),
         ("Can I use Kank-a on my 1-year-old's gums for teething?", ["BENZOCAINE AND MENTHOL UNSPECIFIED FORM"]),
         ("Is Forticept Analgesic Antiseptic Hydrogel okay for a 1-year-old?", ["BENZETHONIUM CHLORIDE LIDOCAINE HYDROCHLORIDE"]),
         ("Who should not use DermaWound, and when should I stop using it?", ["BENZOCAINE HYDROCHLORIDE"]),
         ("Compare the pregnancy warnings of ibuprofen and acetaminophen", ["IBUPROFEN", "ACETAMINOPHEN"]),
         ("What products treat a cold, and how do cold remedies differ from cold sore treatments?", []),
         ("Which topical products help treat diaper rash?", [])]
for q, want in TESTS:
    got = [nm_ for nm_ in dict.fromkeys(re.sub(r"[^A-Z0-9]+", " ", GENERIC_NAME[d].upper()).strip() for d in named_drugs(q))]
    ok = all(w in got for w in want) and (want or not got)
    print("PASS " if ok else "CHECK", "|", q[:60], "| want", want, "| got", got[:5])

PASS  | Compare the indications and purpose of Acetaminophen 500 mg  | want ['ACETAMINOPHEN 500 MG WITH CAFFEINE 65MG', 'ACETAMINOPHEN 500 MG'] | got ['ACETAMINOPHEN 500 MG WITH CAFFEINE 65MG', 'ACETAMINOPHEN 500MG', 'ACETAMINOPHEN 500 MG', 'ACETAMINOPHEN']
PASS  | Can I use Kank-a on my 1-year-old's gums for teething? | want ['BENZOCAINE AND MENTHOL UNSPECIFIED FORM'] | got ['BENZOCAINE AND MENTHOL UNSPECIFIED FORM']
CHECK | Is Forticept Analgesic Antiseptic Hydrogel okay for a 1-year | want ['BENZETHONIUM CHLORIDE LIDOCAINE HYDROCHLORIDE'] | got ['ANTISEPTIC', 'BENZALKONIUM CHLORIDE', 'ALCOHOL']
CHECK | Who should not use DermaWound, and when should I stop using  | want ['BENZOCAINE HYDROCHLORIDE'] | got []
CHECK | Compare the pregnancy warnings of ibuprofen and acetaminophe | want ['IBUPROFEN', 'ACETAMINOPHEN'] | got ['ACETAMINOPHEN IBUPROFEN', 'IBUPROFEN ACETAMINOPHEN', 'ACETAMINOPHEN AND IBUPROFEN']
CHECK | What products treat a cold, and how do cold remedies differ  | want [] | g

In [ ]:
# ===== 6. section choice (embedding similarity, no LLM) =====
SECTION_DESC = {
 "indications_and_usage": "what the product is used for, uses, relieves, treats symptoms",
 "purpose": "purpose of the product", "active_ingredient": "active ingredient and strength",
 "dosage_and_administration": "how much and how often to take or apply, directions, dose by age",
 "dosage_and_administration_table": "dosage table by age or weight",
 "warnings": "warnings, risks, side effects, allergy alert",
 "do_not_use": "do not use, who should not use this product, contraindications",
 "ask_doctor": "ask a doctor before use if you have a medical condition",
 "ask_doctor_or_pharmacist": "ask a doctor or pharmacist before use if you take other drugs",
 "stop_use": "when to stop use and ask a doctor, symptoms last more than days",
 "pregnancy_or_breast_feeding": "pregnant or breast-feeding, ask a health professional",
 "keep_out_of_reach_of_children": "keep out of reach of children, overdose, poison control",
 "storage_and_handling": "how to store the product, storage temperature"}
SEC_LIST = [s for s in SECTION_TYPES if s not in EXCLUDE_SECTIONS]
assert all(c in SECTION_TYPES for c in CORE_SECTIONS), [c for c in CORE_SECTIONS if c not in SECTION_TYPES]
print('section types:', len(SECTION_TYPES), '| used:', len(SEC_LIST))
SEC_EMB = np.array(_embed_call(["search_document: " + SECTION_DESC.get(s, s.replace("_", " ")) for s in SEC_LIST]), dtype="float32")
SEC_EMB /= (np.linalg.norm(SEC_EMB, axis=1, keepdims=True) + 1e-9)
SECTION_ORDER = ["indications_and_usage","purpose","active_ingredient","dosage_and_administration","dosage_and_administration_table",
                 "warnings","do_not_use","ask_doctor","ask_doctor_or_pharmacist","stop_use","pregnancy_or_breast_feeding",
                 "keep_out_of_reach_of_children","storage_and_handling"]
def choose_sections(q, top=3):
    sc = dict(zip(SEC_LIST, SEC_EMB @ qvec(q)))
    ranked = sorted(sc, key=lambda s: -sc[s])
    wanted = set(CORE_SECTIONS) | {s for s in ranked[:top] if s != "storage_and_handling" or sc[s] >= 0.6}
    return wanted, ranked[:top]
print(choose_sections("What should I avoid if I am pregnant and using lidocaine?"))

section types: 14 | used: 13
({'pregnancy_or_breast_feeding', 'do_not_use', 'ask_doctor_or_pharmacist', 'stop_use', 'indications_and_usage', 'warnings'}, ['ask_doctor_or_pharmacist', 'do_not_use', 'pregnancy_or_breast_feeding'])


In [ ]:
# ===== 7. candidate search, rerank, meaning split, cluster check =====
def search_query(q, extra=()):
    idxA = np.concatenate([IDX_BY_SECTION[s] for s in ("indications_and_usage", "purpose") if s in IDX_BY_SECTION])
    fused = defaultdict(float)
    for i, s in hybrid_rank([q], idxA, k=SEARCH_K): fused[i] += s
    for i, s in hybrid_rank([q], None, k=15): fused[i] += 0.5 * s           # all-section safety net
    cand = sorted(fused, key=lambda i: -fused[i])[:RERANK_N]
    cand += [i for i in extra if i not in cand]        # candidates from the original question join each phrase search
    if FLAGS["rerank"] and cand:
        sc = get_reranker().predict([(q, CHUNKS[i]["stamped"]) for i in cand], show_progress_bar=False)
        scores = {i: float(s) for i, s in zip(cand, sc)}
    else: scores = {i: fused.get(i, 0.0) for i in cand}
    return sorted(cand, key=lambda i: -scores[i]), scores

def drugs_from(cand, scores, n):
    best = {}
    for i in cand: best.setdefault(CHUNKS[i]["drug_id"], scores[i])
    out, seen = [], set()
    for d, s in sorted(best.items(), key=lambda x: -x[1]):
        h = label_hash(d)
        if h in seen: continue
        seen.add(h); out.append(d)
        if len(out) >= n: break
    return out

def llm_split(q):
    p = ("Return JSON only. If the question asks about two or three DIFFERENT things (two meanings of one word, or two conditions), "
         "give one short search phrase per thing. Each phrase must keep the main topic words of the question. "
         "If it asks about one thing, return an empty list.\nFormat: {\"phrases\": [\"...\"]}\nQuestion: " + q)
    r = llm_json(p)
    try: ph = [x.strip() for x in r["phrases"] if isinstance(x, str) and x.strip()]
    except Exception: return []
    return ph if 2 <= len(ph) <= 3 else []              # parse failure or more than 3 -> original query only

def cluster_groups(cand, scores):
    top = cand[:30]
    if len(top) < 8: return None
    from sklearn.cluster import KMeans
    lab = KMeans(2, n_init=5, random_state=0).fit(EMB[top]).labels_
    groups = [[i for i, l in zip(top, lab) if l == k] for k in (0, 1)]
    if min(len(g) for g in groups) < 3: return None
    if abs(max(scores[i] for i in groups[0]) - max(scores[i] for i in groups[1])) > CLUSTER_GAP: return None
    ds = [{CHUNKS[i]["drug_id"] for i in g} for g in groups]
    if len(ds[0] & ds[1]) / max(1, min(len(ds[0]), len(ds[1]))) > 0.5: return None
    return groups

def symptom_groups(q):
    phrases = llm_split(q) if FLAGS["split"] == "llm" else []
    cand0, sc0 = search_query(q)                           # the original question is always searched
    if phrases:
        gs = []
        for ph in phrases:                                 # each phrase search also sees the original question's top candidates
            cand, sc = search_query(ph, extra=cand0[:20]); gs.append({"label": ph, "drugs": drugs_from(cand, sc, POOL // len(phrases))})
        return gs, "llm-split"
    if FLAGS["split"] in ("llm", "cluster"):
        cl = cluster_groups(cand0, sc0)
        if cl: return [{"label": f"cluster{k}", "drugs": drugs_from(g, sc0, POOL // 2)} for k, g in enumerate(cl)], "cluster"
    return [{"label": "main", "drugs": drugs_from(cand0, sc0, POOL)}], "single"

In [ ]:
# ===== 8. quota, blocks, ordering, prompt (3 few-shot examples + optional abstention shot) =====
SHOTS = """### EXAMPLE 1
Question: What can I take for fatigue with pale skin, what does the label say about pregnancy, and can it be taken with food?
Context:
### DIGITALIS
NOTE: no text for pregnancy_or_breast_feeding.
[S1] (indications_and_usage) For the temporary relief of symptoms including: \u2022 stomach support \u2022 vomiting \u2022 spleen support \u2022 stomach pain \u2022 loss of appetite \u2022 loose stools \u2022 abdominal bloating \u2022 fluid retention These statements are based upon homeopathic principles. They have not been reviewed by the Food and Drug Administration

### DIEFFENBACHIA SEGUINE
[S2] (indications_and_usage) To reduce withdrawal symptoms, including cravings to smoke, withdrawal nerves, and increased appetite
[S3] (pregnancy_or_breast_feeding) If pregnant or breast-feeding, ask a health professional before use

### BENZENE
[S4] (indications_and_usage) See symptoms on front panel ; Relieves fatigue with paleness *
[S5] (pregnancy_or_breast_feeding) If pregnant or breast-feeding, ask a health professional before use
[S6] (dosage_and_administration) Adults and children: At the onset of symptoms, dissolve 5 pellets under the tongue 3 times a day until symptoms are relieved or as directed by a doctor
SCRATCHPAD:
1. Need: A) product for fatigue with pale skin. B) its pregnancy text. C) food instructions.
2. DIGITALIS: stomach and spleen symptoms, no fatigue. DIEFFENBACHIA: smoking withdrawal, so its pregnancy line [S3] belongs to a different product. Both left out.
3. BENZENE: A [S4]. B [S5]. C: [S6] does not mention food.
4. Answer: Benzene only, one tag per claim, missing items as Gap lines.
ANSWER:
- Benzene: relieves fatigue with paleness [S4].
- Benzene: if pregnant or breast-feeding, ask a health professional before use [S5].
- Gap: the retrieved text says nothing about taking it with food.

### EXAMPLE 2
Question: What products help treat buildup, depending on whether it is flaky scalp buildup or facial milia buildup, and when should each be stopped?
Context:
### DHS SAL
[S1] (indications_and_usage) Controls crusty, flaky buildup, while relieving scalp itching and flaking, symptomatic of Psoriasis and Seborrheic Dermatitis
[S2] (warnings) For external use only ; Ask a physician before use if you have Psoriasis or Seborrheic Dermatitis that covers a large area of the body ; Do not get into eyes. If contact occurs, rinse eyes thoroughly with water ; Stop use and ask a physician if condition worsens or does not improve after regular use

### DIDECYL DIMETHYL AMMONIUM CHLORIDE
[S3] (indications_and_usage) Fight and reduce signs of milia ; Help to resolve acne skin problems
[S4] (stop_use) Stop using and ask a doctor if rash or irritation happens
SCRATCHPAD:
1. Need: two conditions, A) flaky scalp buildup and B) facial milia buildup. For each: a product and a stop rule.
2. A: DHS SAL [S1]. Stop rule in its warnings [S2].
3. B: Didecyl dimethyl ammonium chloride [S3]. Stop rule [S4]. The label never uses the word "buildup".
4. Answer: one bullet per condition, one tag per claim.
ANSWER:
- Flaky scalp buildup: DHS SAL controls crusty, flaky buildup and relieves scalp itching and flaking [S1]. Stop use and ask a physician if the condition worsens or does not improve after regular use [S2].
- Facial milia buildup: Didecyl dimethyl ammonium chloride is labelled to fight and reduce signs of milia [S3]. Stop using and ask a doctor if rash or irritation happens [S4].

### EXAMPLE 3
Question: Compare the do-not-use conditions and stop-use rules of Advanced Scar Gel and Diethyltoluamide.
Context:
### ADVANCED SCAR GEL
[S1] (do_not_use) Do Not Use on deep or puncture wounds animal bites serious burns
[S2] (stop_use) Stop Use and Ask a Doctor if condition worsens symptoms last for more than 7 days or clears up and occur again within a few days

### DIETHYLTOLUAMIDE
NOTE: no text for stop_use.
[S3] (do_not_use) do not use for the following people: infants under six months ; do not use excessive amounts or for long periods of time ; do not use it in an enclosed area
SCRATCHPAD:
1. Need: compare A) do-not-use and B) stop-use for two drugs.
2. SCAR GEL: A [S1]. B [S2].
3. DIETHYLTOLUAMIDE: A [S3]. B: the NOTE says no stop_use text.
4. Answer: one bullet per drug and point. Keep each drug's claims separate.
ANSWER:
- Advanced Scar Gel, do not use: on deep or puncture wounds, animal bites or serious burns [S1].
- Advanced Scar Gel, stop use: stop and ask a doctor if the condition worsens, symptoms last more than 7 days, or they clear up and return within a few days [S2].
- Diethyltoluamide, do not use: for infants under six months, in excessive amounts or for long periods, or in an enclosed area [S3].
- Gap: Diethyltoluamide has no stop-use text in the retrieved label.
"""
SHOT4 = """
### EXAMPLE 4
Question: Can DHS SAL be used during pregnancy?
Context:
### DHS SAL
NOTE: no text for pregnancy_or_breast_feeding.
[S1] (warnings) For external use only ; Ask a physician before use if you have Psoriasis or Seborrheic Dermatitis that covers a large area of the body ; Do not get into eyes. If contact occurs, rinse eyes thoroughly with water ; Stop use and ask a physician if condition worsens or does not improve after regular use
SCRATCHPAD:
1. Need: pregnancy text for DHS SAL.
2. The NOTE says no pregnancy text, and [S1] does not mention pregnancy.
3. Answer: gap only, no outside knowledge.
ANSWER:
- Gap: the retrieved DHS SAL text says nothing about pregnancy.
"""
RULES = ("You answer ONLY from the FDA drug label excerpts in the Context. Never use outside knowledge.\n"
         "- Think briefly under SCRATCHPAD:, then write ANSWER:.\n"
         "- Each claim is one bullet that names the drug and ends with its source tag, e.g. [S2].\n"
         "- Leave out drugs that do not match the question. Keep each drug's facts separate.\n"
         "- If something asked is not in the text, write a bullet starting with 'Gap:' and no tag.\n")

SECTION_RANK = {s: i for i, s in enumerate(SECTION_ORDER)}
def build_block(d, wanted, missing_note, qvs, quota, tagc, sources):
    idxs = [i for i in IDX_BY_DRUG.get(d, []) if CHUNKS[i]["section"] in wanted]
    idxs = sorted(idxs, key=lambda i: -max(float(EMB[i] @ v) for v in qvs))
    chosen, used, seen = [], 0, set()
    for i in idxs:                                        # best chunk of every wanted section first
        s = CHUNKS[i]["section"]
        if s not in seen and used + len(CHUNKS[i]["text"]) <= quota: chosen.append(i); seen.add(s); used += len(CHUNKS[i]["text"])
    per_sec = Counter(CHUNKS[i]["section"] for i in chosen)
    for i in idxs:                                        # then fill the rest of the quota (max MAX_CHUNKS_PER_SECTION per section)
        sec = CHUNKS[i]["section"]
        if i not in chosen and per_sec[sec] < MAX_CHUNKS_PER_SECTION and used + len(CHUNKS[i]["text"]) <= quota:
            chosen.append(i); used += len(CHUNKS[i]["text"]); per_sec[sec] += 1
    chosen.sort(key=lambda i: (SECTION_RANK.get(CHUNKS[i]["section"], 99), i))
    have = {CHUNKS[i]["section"] for i in IDX_BY_DRUG.get(d, [])}
    lines = [f"### {GENERIC_NAME[d]}"]
    miss = [s for s in missing_note if s not in have]
    if miss: lines.append("NOTE: no text for " + ", ".join(miss) + ".")
    for i in chosen:
        tagc[0] += 1; tag = f"S{tagc[0]}"
        sources.append({"tag": tag, "drug_id": d, "drug": GENERIC_NAME[d], "section": CHUNKS[i]["section"], "node": CHUNKS[i]["cid"], "text": CHUNKS[i]["text"]})
        lines.append(f"[{tag}] ({CHUNKS[i]['section']}) {CHUNKS[i]['text']}")
    return "\n".join(lines)

def order_drugs(ds):
    if len(ds) >= 6: return [ds[0]] + ds[2:] + [ds[1]]     # best first, second-best last, never the best in the middle
    return ds

def build_context(q, groups_drugs, shrink=1.0, labels=None):
    wanted, top3 = choose_sections(q); qv = qvec(q)
    total = int(CONTEXT_TOKENS * CHARS_PER_TOKEN * shrink)
    live = [(g, lb) for g, lb in zip(groups_drugs, labels or [None] * len(groups_drugs)) if g]
    group_budget = total // max(1, len(live))              # budget per meaning group first
    tagc, sources, blocks = [0], [], []
    for g, lb in live:
        quota = min(group_budget // len(g), 9000)          # then per drug inside the group
        qvs = [qv] + ([qvec(lb)] if lb else [])            # chunks scored by the whole question AND the group's own phrase
        for d in order_drugs(g): blocks.append(build_block(d, wanted, top3, qvs, quota, tagc, sources))
    return "\n\n".join(blocks), sources

def build_prompt(q, context):
    shots = (SHOTS + (SHOT4 if FLAGS["shot4"] else "")) if FLAGS["few_shot"] else ""
    return (RULES + "\n" + shots + f"\n### NOW ANSWER\nQuestion: {q}\nContext:\n{context}\n\nQuestion again: {q}\nSCRATCHPAD:\n")

In [ ]:
# ===== 9. answer + verify (flag only) =====
def parse_answer(raw):
    body = raw.split("ANSWER:")[-1] if "ANSWER:" in raw else raw           # scratchpad is stripped here
    claims = []
    for line in body.splitlines():
        line = line.strip()
        if not line.startswith(("-", "*", "\u2022")): continue
        text = line.lstrip("-*\u2022 ").strip()
        tags = []
        for a, b in re.findall(r"\[S(\d+)(?:\s*[-\u2013]\s*S?(\d+))?\]", text):
            tags += [f"S{n}" for n in range(int(a), int(b) + 1)] if b else [f"S{a}"]
        claims.append({"claim": re.sub(r"\[S[\d\s\-\u2013S]*\]", "", text).strip(), "tags": list(dict.fromkeys(tags)), "gap": text.lower().startswith("gap:")})
    return claims

CSTOP = STOP | {"this","that","from","have","with","text","label","retrieved","product","use","used"}
def overlap(claim, ev):
    cw = {w for w in re.findall(r"[a-z0-9]+", claim.lower()) if len(w) >= 4 and w not in CSTOP}
    ew = set(re.findall(r"[a-z0-9]+", ev.lower()))
    return (len(cw & ew) / len(cw)) if cw else 1.0

def verify(claims, sources):
    by = {s["tag"]: s for s in sources}
    for c in claims:
        if c["gap"]: c["status"] = "gap"; continue                        # gap lines are allowed, not checked
        if not c["tags"]: c["status"] = "no_citation"; continue
        if any(t not in by for t in c["tags"]): c["status"] = "invalid_citation"; continue
        ev = "\n".join(by[t]["text"] for t in c["tags"])
        if overlap(c["claim"], ev) >= 0.8: c["status"] = "supported"; continue     # code check first
        v = llm(f"Label text:\n{ev}\n\nClaim: {c['claim']}\n\nDoes the label text directly support the claim? Answer with one word: YES or NO.",
                temperature=0.0, num_predict=3)
        c["status"] = "supported" if v.upper().startswith("YES") else "unsupported"
    return claims

def answer_question(q, verbose=True):
    t0 = time.time(); named = named_drugs(q); hits = list(LAST_HITS)
    if named: mode, groups, groups_drugs, how = "drug", [{"label": "named", "drugs": named}], [named], "named"
    else:
        groups, how = symptom_groups(q); mode = "symptom"
        k = max(1, TOP_PASS // len(groups)); groups_drugs = [g["drugs"][:k] for g in groups]
    if not any(groups_drugs):
        rec = {"question": q, "mode": mode, "drugs": [], "sources": [], "claims": [], "abstained": True, "abstain_reason": "no chunks retrieved",
               "final_answer": ABSTAIN_TEXT, "raw_answer": ABSTAIN_TEXT, "seconds": round(time.time() - t0, 1), "sections_in_context": []}
        return rec
    shrink = 1.0
    while True:                                           # keep prompt + answer inside NUM_CTX (about 3.5 chars per token)
        context, sources = build_context(q, groups_drugs, shrink, [g["label"] if how == "llm-split" else None for g in groups])
        prompt = build_prompt(q, context)
        if len(prompt) / 3.5 + ANSWER_TOKENS <= NUM_CTX or shrink < 0.3: break
        shrink *= 0.8
    raw = llm(prompt); ptoks = LAST_PROMPT_TOKENS[0]
    claims = parse_answer(raw)
    if FLAGS["verify"]: claims = verify(claims, sources)
    real = [c for c in claims if not c["gap"]]
    rate = round(sum(c.get("status") != "supported" for c in real) / len(real), 3) if real else None
    final = "\n".join(("- " if c.get("status") in ("supported", "gap", None) else f"- [FLAGGED: {c['status']}] ") + c["claim"] + " " + "".join(f"[{t}]" for t in c["tags"]) for c in claims) or raw
    rec = {"question": q, "mode": mode, "split": how, "entity_hits": hits, "groups": [{"label": g["label"], "drugs": [GENERIC_NAME[d] for d in g["drugs"]]} for g in groups],
           "drugs": [GENERIC_NAME[d] for g in groups_drugs for d in g], "sections_in_context": sorted({s["section"] for s in sources}),
           "sources": [{k: v for k, v in s.items() if k not in ("text", "drug_id")} for s in sources], "context_chars": len(context),
           "prompt_tokens": ptoks, "claims": claims, "unsupported_rate": rate, "raw_answer": raw, "final_answer": final,
           "abstained": not real, "abstain_reason": "answer has only Gap lines" if not real else None, "seconds": round(time.time() - t0, 1)}
    if verbose: print(f"\nQ: {q}\n mode={mode}/{how} drugs={rec['drugs']} prompt_tokens={ptoks}\n{final}\n ({rec['seconds']}s)")
    return rec

for q in ["What should I avoid if I am pregnant and using lidocaine?",
          "Can I use Kank-a on my 1-year-old's gums for teething?",
          "Compare the indications and purpose of Acetaminophen 500 mg with Caffeine 65 mg versus Acetaminophen 500 mg."]:
    answer_question(q); print("-" * 70)


Q: What should I avoid if I am pregnant and using lidocaine?
 mode=drug/named drugs=['LIDOCAINE', 'LIDOCAINE HYDROCHLORIDE', 'LIDOCAINE 4%'] prompt_tokens=2116
- Gap: No specific text in the retrieved label advises what to avoid if pregnant and using lidocaine. 
 (4.3s)
----------------------------------------------------------------------

Q: Can I use Kank-a on my 1-year-old's gums for teething?
 mode=drug/named drugs=['BENZOCAINE AND MENTHOL, UNSPECIFIED FORM'] prompt_tokens=1775
- Gap: the retrieved text says "Do not use for teething in children under 2 years of age" , so it is not suitable for a 1-year-old's gums for teething. [S5]
 (3.6s)
----------------------------------------------------------------------

Q: Compare the indications and purpose of Acetaminophen 500 mg with Caffeine 65 mg versus Acetaminophen 500 mg.
 mode=drug/named drugs=['ACETAMINOPHEN 500 MG WITH CAFFEINE 65MG', 'ACETAMINOPHEN 500MG', 'ACETAMINOPHEN 500 MG', 'ACETAMINOPHEN'] prompt_tokens=3636
- Acetaminop

In [ ]:
# ===== 10. entity check on the QA set (no LLM): is the gold drug found by name/alias? =====
QA = json.load(open(QA_PATH))
nm = lambda s: re.sub(r"[^A-Z0-9]+", " ", s.upper()).strip()
gold_drugs = lambda qa: [m for t in qa["trace"] for m in re.findall(r"DRUG: (.*?) \| SOURCE", t)]
rows = []
for qid, qa in QA.items():
    got = {nm(GENERIC_NAME[d]) for d in named_drugs(qa["question"])}
    gold = [nm(g) for g in gold_drugs(qa)]
    rows.append({"qid": qid, "named_found": len(got), "gold_hit": sum(g in got for g in gold), "gold_n": len(gold)})
ec = pd.DataFrame(rows); ec["type"] = ec.qid.str.split("-").str[0]
print(ec.groupby("type").agg(n=("qid", "count"), named_found=("named_found", "mean"), gold_hit=("gold_hit", "sum"), gold_n=("gold_n", "sum")).round(2))
print("questions with named drugs found but NO gold hit (possible false matches):", list(ec[(ec.named_found > 0) & (ec.gold_hit == 0)].qid))

      n  named_found  gold_hit  gold_n
type                                  
T1    6         2.00         4       6
T2    8         1.62         0      11
T3    9         1.22         6       9
T4    9         0.78         0      18
T5    9         3.22        14      18
T6    9         1.33         9       9
T7    5         2.40         5       5
T8    5         2.40         4       8
questions with named drugs found but NO gold hit (possible false matches): ['T1-001', 'T1-002', 'T2-003', 'T2-004', 'T2-005', 'T2-006', 'T2-008', 'T4-001', 'T4-007', 'T4-009', 'T8-001', 'T8-003', 'T3-003']


In [ ]:
# ===== 11. run the QA set (resumable, saved after every question) =====
RESET = True      # True: back up old results and start fresh. Set False to resume after a restart.
if RESET and "RESET_DONE" not in globals() and os.path.exists(RESULTS_PATH):
    os.rename(RESULTS_PATH, RESULTS_PATH.replace(".json", f"_old_{int(time.time())}.json"))
RESET_DONE = True
SKIP_WORDS = {"and","the","with","mg","unspecified","form","tablet","tablets","cream","ointment","gel","solution","topical","oral","liquid"}
ing = lambda s: frozenset(t for t in re.findall(r"[a-z]{3,}", s.lower()) if t not in SKIP_WORDS)
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
todo = [q for q in QA if q not in RESULTS]; print(f"done {len(RESULTS)} | to run {len(todo)}")
for n, qid in enumerate(todo, 1):
    qa = QA[qid]; print(f"\n######## {qid} ({n}/{len(todo)})")
    try: rec = answer_question(qa["question"])
    except Exception as e:
        print("!! ERROR", qid, repr(e)); traceback.print_exc(); continue
    gold_raw = gold_drugs(qa); got = [nm(d) for d in rec["drugs"]]; got_ing = {ing(d) for d in rec["drugs"]}
    rec["gold_drugs"] = [nm(g) for g in gold_raw]
    rec["drug_recall"] = round(sum(nm(g) in got for g in gold_raw) / len(gold_raw), 2) if gold_raw else None
    rec["drug_recall_ing"] = round(sum(ing(g) in got_ing for g in gold_raw) / len(gold_raw), 2) if gold_raw else None
    rec["section_recall"] = round(len(set(qa["sections"]) & set(rec["sections_in_context"])) / len(qa["sections"]), 2) if qa["sections"] else None
    rec["answerable"], rec["answer_key"] = qa["answerable"], qa["answer"]
    rec["abstain_correct"] = (rec["abstained"] == (not qa["answerable"]))
    RESULTS[qid] = rec; json.dump(RESULTS, open(RESULTS_PATH, "w"), indent=2)
print("saved", len(RESULTS), "->", RESULTS_PATH)

done 0 | to run 60

######## T1-001 (1/60)

Q: What should I avoid if I am pregnant and using lidocaine?
 mode=drug/named drugs=['LIDOCAINE', 'LIDOCAINE HYDROCHLORIDE', 'LIDOCAINE 4%'] prompt_tokens=2116
- Gap: No specific text in the retrieved label for what to avoid if pregnant and using lidocaine. 
 (3.9s)

######## T1-002 (2/60)

Q: What organ-specific warning and daily intake limit apply to Acetaminophen and Phenylephrine HCl, and what medical conditions require consulting a doctor before use?
 mode=drug/named drugs=['ACETAMINOPHEN PHENYLEPHRINE HCL', 'ACETAMINOPHEN AND PHENYLEPHRINE HCL'] prompt_tokens=2642
- Organ-specific warning: liver warning . [S10]
- [FLAGGED: no_citation] Daily intake limit: none mentioned. 
- Medical conditions that require consulting a doctor before use: heart disease, thyroid disease, diabetes, liver disease, high blood pressure, difficulty in urination due to enlargement of the prostate gland , . [S5][S13]
- Gap: no mention of specific organ or daily i

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]


Q: What over-the-counter oral pain relievers are indicated for headaches, and what specific warnings or precautions apply to their use during pregnancy?
 mode=symptom/llm-split drugs=['PETROLEUM', 'DIVALPROEX SODIUM', 'ARNICA MONTANA, CAULOPHYLLUM THALICTROIDES, CIMICIFUGA RACEMOSA, GELSEMIUM SEMPERVIRENS, PULSATILLA.', 'APIS MEL., BERBER. VULG., BRYONIA, CANTHARIS, CINCHONA, EQUISETUM HYEMALE, FERRUM PHOSPHORICUM, GELSEMIUM, JUNIPERUS COM., PAREIRA, PHOSPHORUS, PLANTAGO, PULSATILLA, SABAL, TEREBINTHINA, UVA-URSI, ECHINACEA, HYDRASTIS, LAPPA, SOLIDAGO', 'NORETHINDRONE', 'HYDROGEN PEROXIDE, MENTHOL'] prompt_tokens=3126
- [FLAGGED: no_citation] PETROLEUM: no text for headache indication. 
- [FLAGGED: no_citation] DIVALPROEX SODIUM: not an oral pain reliever, but an epilepsy medication. 
- [FLAGGED: no_citation] ARNICA MONTANA, CAULOPHYLLUM THALICTROIDES, CIMICIFUGA RACEMOSA, GELSEMIUM SEMPERVIRENS, PULSATILLA: no text for headache indication. 
- [FLAGGED: no_citation] APIS MEL., BERBER.

In [ ]:
# ===== 12. summary by test type (automatic metrics; hand-score final_answer vs answer_key with your rule) =====
df = pd.DataFrame([{"qid": k, "type": k.split("-")[0], **{x: v.get(x) for x in
      ["drug_recall", "drug_recall_ing", "section_recall", "abstain_correct", "unsupported_rate", "prompt_tokens", "seconds"]}} for k, v in RESULTS.items()])
print(df.groupby("type").agg(n=("qid", "count"), drug_recall=("drug_recall", "mean"), drug_recall_ing=("drug_recall_ing", "mean"),
      section_recall=("section_recall", "mean"), abstain_correct=("abstain_correct", "mean"), unsupported_rate=("unsupported_rate", "mean"),
      max_prompt_tokens=("prompt_tokens", "max"), avg_seconds=("seconds", "mean")).round(2))
print("\nOverall drug_recall:", round(df.drug_recall.mean(), 2), "| ingredient:", round(df.drug_recall_ing.mean(), 2),
      "| section_recall:", round(df.section_recall.mean(), 2), "| abstain_correct:", round(df.abstain_correct.mean(), 2))
print("Max prompt tokens seen:", df.prompt_tokens.max(), "of NUM_CTX", NUM_CTX, "-> raise CONTEXT_TOKENS or NUM_CTX only if there is margin")
print("Gold drug not retrieved (ingredient match):", list(df[df.drug_recall_ing < 1].qid))

      n  drug_recall  drug_recall_ing  section_recall  abstain_correct  \
type                                                                     
T1    6         0.67             0.67            0.94             0.83   
T2    8         0.00             0.00            0.77             1.00   
T3    9         0.67             0.67            0.94             0.67   
T4    9         0.11             0.11            0.89             0.89   
T5    9         0.78             0.78            0.83             0.89   
T6    9         1.00             1.00             NaN             0.89   
T7    5         1.00             1.00            1.00             0.80   
T8    5         0.60             0.60            0.79             1.00   

      unsupported_rate  max_prompt_tokens  avg_seconds  
type                                                    
T1                0.07               2642         6.70  
T2                0.12               4475        48.65  
T3                0.11         